# Capitals relation-feature pilot

This add-on tests whether graph-derived SAE features causally support the **capital relation**, rather than merely one capital name. It uses the existing capitals SAEs, ranks features on discovery countries, and confirms a frozen Top-3 panel on disjoint countries. The primary outcome is a selective change in `logit(capital) - logit(country)`; a categorical country-name flip is recorded but is not assumed.

## 1. Mount Drive and update the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess
from pathlib import Path

repo_url = 'https://github.com/evey-dev/test_run.git'
repo_dir = Path('/content/test_run')
if (repo_dir / '.git').exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Working directory:', os.getcwd())

## 2. Install the project

In [ ]:
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .

## 3. Restore the existing capitals SAE checkpoints

In [ ]:
import shutil

layers = [4, 8, 12, 16, 20, 24, 28]
drive_sae_dir = Path('/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train')
local_sae_dir = Path('mechanistic_data/sae_checkpoints_capitals_final_train')
required = [item for layer in layers for item in (f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json')]
missing = [name for name in required if not (drive_sae_dir / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing capitals checkpoints in Drive: {missing}')
local_sae_dir.mkdir(parents=True, exist_ok=True)
for name in required:
    shutil.copy2(drive_sae_dir / name, local_sae_dir / name)
print('Restored', len(required), 'checkpoint files to', local_sae_dir)

## 4. Build one contrastive candidate graph

The graph is built on Zarqa, with `Amman` as the capital target and `Jordan` as the country contrast. Jordan is excluded from both later splits.

In [ ]:
!mkdir -p outputs/capitals_relation_pilot
!python -u src/attribution_graph.py \
  --prompt "Fact: The capital of the country containing Zarqa is named" \
  --target "Amman" \
  --contrast-target "Jordan" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --top-k-nodes 25 --top-k-edges 35 \
  --output-json outputs/capitals_relation_pilot/capitals_amman_over_jordan_graph.json \
  --output-html outputs/capitals_relation_pilot/capitals_amman_over_jordan_graph.html \
  --output-mermaid outputs/capitals_relation_pilot/capitals_amman_over_jordan_graph.md

## 5. Discover and confirm a relation-selective panel

This is resumable: rerunning the cell continues from the JSON checkpoint. Expect many forward passes because each graph feature is tested on both a capital prompt and its inverse-country control.

In [ ]:
!python -u src/capitals_relation_feature_screen.py \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --capitals-csv data/capitals_data.csv \
  --graph outputs/capitals_relation_pilot/capitals_amman_over_jordan_graph.json \
  --exclude-country Jordan \
  --discovery-cases 8 \
  --confirmation-cases 16 \
  --panel-sizes 1 3 5 10 20 \
  --primary-panel-size 3 \
  --random-panels 5 \
  --output outputs/capitals_relation_pilot/capitals_relation_feature_screen.json

## 6. Print the result that is safe to claim

In [ ]:
import json

result_path = Path('outputs/capitals_relation_pilot/capitals_relation_feature_screen.json')
result = json.loads(result_path.read_text())
primary = result['confirmation']['primary_result']
summary = primary['summary']
print('Status:', result['status'])
print('Candidate graph features:', result['candidate_feature_count'])
print('Frozen Top-3:', result['discovery']['frozen_feature_order'][:3])
print('Capital-prompt delta:', summary['mean_capital_prompt_delta'])
print('Inverse-country delta:', summary['mean_inverse_country_prompt_delta'])
print('Relation-specific difference:', summary['mean_relation_specific_difference'])
print('95% CI:', summary['bootstrap_95_ci_mean_relation_specific_difference'])
print('Capital-to-country flips:', summary['capital_to_country_flip_fraction'])
print('Confirmatory rule passed:', primary['supports_capital_relation_selectivity_under_predeclared_rule'])
print()
if primary['supports_capital_relation_selectivity_under_predeclared_rule']:
    print('Presentation wording: a compact panel selectively supports the capital relation on held-out countries.')
else:
    print('Presentation wording: no compact capital-relation panel was confirmed; do not label the features as a capital panel.')

## 7. Save the complete pilot output to Drive

In [ ]:
drive_output = Path('/content/drive/MyDrive/mphil-project/outputs/capitals_relation_pilot')
drive_output.mkdir(parents=True, exist_ok=True)
for source in Path('outputs/capitals_relation_pilot').iterdir():
    if source.is_file():
        shutil.copy2(source, drive_output / source.name)
print('Saved pilot outputs to', drive_output)
!ls -lh /content/drive/MyDrive/mphil-project/outputs/capitals_relation_pilot